In [5]:

# ==============================================================================
# === CELL 1: Configuration, Environment Auto-Detection & Logging
# ==============================================================================

import os, sys, gc, re, time, warnings, json
import numpy as np
import pandas as pd
from collections import defaultdict
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# Auto-detect Environment: AWS EC2 vs Kaggle vs Local
# ------------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle")
IS_AWS = os.path.exists("/home/ubuntu") or not IS_KAGGLE

if IS_KAGGLE:
    ENV_NAME = "Kaggle (2x T4 GPU, 30GB RAM)"
    KAGGLE_INPUT = "/kaggle/input/datasets/neuronbytes/amazon-ml-challenge-2026"
    if os.path.exists(os.path.join(KAGGLE_INPUT, "train")):
        TRAIN_DIR = os.path.join(KAGGLE_INPUT, "train")
        TEST_DIR = os.path.join(KAGGLE_INPUT, "test")
    elif os.path.exists(os.path.join(KAGGLE_INPUT, "dataset", "train")):
        TRAIN_DIR = os.path.join(KAGGLE_INPUT, "dataset", "train")
        TEST_DIR = os.path.join(KAGGLE_INPUT, "dataset", "test")
    else:
        TRAIN_DIR = KAGGLE_INPUT
        TEST_DIR = KAGGLE_INPUT
    OUTPUT_DIR = "/kaggle/working"
else:
    # AWS / Local directory structure
    ENV_NAME = "AWS EC2 / Standalone Server"
    BASE_DIR = os.path.expanduser("~/data") if os.path.exists(os.path.expanduser("~/data")) else os.getcwd()
    TRAIN_DIR = os.path.join(BASE_DIR, "train") if os.path.exists(os.path.join(BASE_DIR, "train")) else BASE_DIR
    TEST_DIR = os.path.join(BASE_DIR, "test") if os.path.exists(os.path.join(BASE_DIR, "test")) else BASE_DIR
    OUTPUT_DIR = os.path.join(BASE_DIR, "output_v4")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hardware resources
NUM_CPUS = os.cpu_count() or 4
print("=" * 80)
print(f"🚀 PIPELINE V4 INITIALIZING ON: {ENV_NAME}")
print(f"   CPUs Available: {NUM_CPUS} cores")
print(f"   Train Directory: {TRAIN_DIR}")
print(f"   Test Directory:  {TEST_DIR}")
print(f"   Output & Checkpoints: {OUTPUT_DIR}")
print("=" * 80)


def fmt_time(seconds):
    """Format seconds into human-readable string: 1h 05m 20s or 2m 15s or 45s."""
    if seconds is None or seconds < 0:
        return "0s"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    elif m > 0:
        return f"{m}m {s:02d}s"
    return f"{s}s"


🚀 PIPELINE V4 INITIALIZING ON: Kaggle (2x T4 GPU, 30GB RAM)
   CPUs Available: 4 cores
   Train Directory: /kaggle/input/datasets/neuronbytes/amazon-ml-challenge-2026
   Test Directory:  /kaggle/input/datasets/neuronbytes/amazon-ml-challenge-2026
   Output & Checkpoints: /kaggle/working


In [6]:


# ==============================================================================
# === CELL 2: Package Verification & High-Performance Dependencies
# ==============================================================================

import subprocess

required_pkgs = [
    "rapidfuzz", "sparse_dot_topn", "unidecode", "lightgbm",
    "sentence-transformers", "torch"
]

print("Checking dependencies...")
for pkg in required_pkgs:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"  Installing missing package: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Try installing faiss-gpu or faiss-cpu
try:
    import faiss
except ImportError:
    print("  Installing FAISS...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu"])
        import faiss
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        import faiss

import torch
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from unidecode import unidecode
import lightgbm as lgb
from sentence_transformers import SentenceTransformer

# Sparse matrix multiplication top-n compatibility
try:
    from sparse_dot_topn import sp_matmul_topn
    USE_NEW_API = True
    print("  ✅ Using sparse_dot_topn.sp_matmul_topn (v1.0+ API)")
except ImportError:
    from sparse_dot_topn import awesome_cossim_topn
    USE_NEW_API = False
    print("  ✅ Using sparse_dot_topn.awesome_cossim_topn (legacy API)")

# Check GPU capability
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
print(f"  PyTorch Version: {torch.__version__} | CUDA: {CUDA_AVAILABLE} (Found {GPU_COUNT} GPUs)")
for i in range(GPU_COUNT):
    print(f"    GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

FAISS_GPUS = faiss.get_num_gpus() if hasattr(faiss, 'get_num_gpus') else 0
print(f"  FAISS GPU acceleration: {'✅ Available (' + str(FAISS_GPUS) + ' GPUs)' if FAISS_GPUS > 0 else '⚠️ CPU fallback'}")
print("✅ All high-performance libraries verified!")



Checking dependencies...
  ✅ Using sparse_dot_topn.sp_matmul_topn (v1.0+ API)
  PyTorch Version: 2.10.0+cu128 | CUDA: True (Found 2 GPUs)
    GPU 0: Tesla T4 (15.6 GB)
    GPU 1: Tesla T4 (15.6 GB)
  FAISS GPU acceleration: ✅ Available (2 GPUs)
✅ All high-performance libraries verified!


In [7]:

# ==============================================================================
# === CELL 3: Robust Multilingual Preprocessing & Normalization
# ==============================================================================

def find_file(filename):
    """Search for data files across multiple potential directories."""
    candidates = [
        os.path.join(TRAIN_DIR, filename),
        os.path.join(TEST_DIR, filename),
        os.path.join(OUTPUT_DIR, filename),
        filename
    ]
    if IS_KAGGLE:
        candidates.append(os.path.join("/kaggle/input", filename))
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Cannot find {filename} in {candidates}")


def normalize_name(name):
    """
    Standardize business names:
    - Normalizes legal suffixes (US, India, and France)
    - Strips punctuation and leading noise characters
    - Standardizes ampersands and spacing
    """
    if pd.isna(name) or str(name).strip() == "":
        return ""
    s = unidecode(str(name)).lower().strip()
    s = re.sub(r'^[-\s\.\*#]+', '', s)  # Remove noise prefixes like '-- '
    
    # Standardize business suffixes (including US, UK, Indian, French legal entities)
    legal_patterns = [
        (r'\bincorporated\b', 'inc'), (r'\bcorporation\b', 'corp'), (r'\blimited\b', 'ltd'),
        (r'\bprivate\b', 'pvt'), (r'\bcompany\b', 'co'), (r'\bllc\b', 'llc'),
        (r'\bl\.l\.c\.?\b', 'llc'), (r'\bllp\b', 'llp'), (r'\bl\.l\.p\.?\b', 'llp'),
        (r'\binc\.?\b', 'inc'), (r'\bltd\.?\b', 'ltd'), (r'\bpvt\.?\b', 'pvt'),
        (r'\bcorp\.?\b', 'corp'), (r'\bco\.?\b', 'co'),
        # French entity types (seen in test set!)
        (r'\bs\.?a\.?s\.?u\.?\b', 'sasu'), (r'\bs\.?a\.?s\.?\b', 'sas'),
        (r'\bs\.?a\.?r\.?l\.?\b', 'sarl'), (r'\be\.?u\.?r\.?l\.?\b', 'eurl'),
        (r'\bs\.?a\.?\b', 'sa'), (r'\bs\.?c\.?i\.?\b', 'sci'),
    ]
    for pat, rep in legal_patterns:
        s = re.sub(pat, rep, s)
    s = s.replace('&', ' and ')
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_address(addr):
    """
    Standardize business addresses:
    - Expands common road and suite abbreviations
    - Strips 'null', 'nan', 'none'
    - Normalizes whitespace
    """
    if pd.isna(addr) or str(addr).strip().lower() in ("", "nan", "null", "none", "n/a"):
        return ""
    s = unidecode(str(addr)).lower().strip()
    addr_patterns = [
        (r'\bst\b', 'street'), (r'\brd\b', 'road'), (r'\bave\b', 'avenue'),
        (r'\bblvd\b', 'boulevard'), (r'\bdr\b', 'drive'), (r'\bct\b', 'court'),
        (r'\bln\b', 'lane'), (r'\bpkwy\b', 'parkway'), (r'\bhwy\b', 'highway'),
        (r'\bapt\b', 'apartment'), (r'\bste\b', 'suite'),
    ]
    for pat, rep in addr_patterns:
        s = re.sub(pat, rep, s)
    s = re.sub(r'\bnull\b', '', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def preprocess_df(df, name=""):
    """Vectorized text cleaning and dual-representation preparation."""
    t0 = time.time()
    print(f"  Preprocessing {name} ({len(df):,} records)...", end=" ", flush=True)
    df = df.copy()
    
    # PDF says country is an OPEN SET -- do NOT hardcode {US, India}.
    # France appears only in test. Fill any missing country with 'Unknown' to avoid NaN partition errors.
    if 'country' in df.columns:
        df['country'] = df['country'].fillna('Unknown').astype(str).str.strip()
    else:
        df['country'] = 'Unknown'
    
    # 1. Cleaned ASCII text for TF-IDF & Fuzzy matching
    df['name_norm'] = df['business_name'].apply(normalize_name)
    df['addr_norm'] = df['business_address'].apply(normalize_address)
    df['name_addr'] = (df['name_norm'] + ' ' + df['addr_norm']).str.strip()
    
    # 2. Raw original text for Multilingual Embedding (retains Hindi, Tamil, French scripts)
    raw_name = df['business_name'].fillna('').astype(str).str.strip()
    raw_addr = df['business_address'].fillna('').astype(str).str.strip()
    df['embed_text'] = (raw_name + ' ' + raw_addr).str.strip()
    
    print(f"done in {time.time()-t0:.1f}s")
    return df


print("\nLoading training datasets...")
t_load = time.time()

train_s1 = preprocess_df(pd.read_csv(find_file("train_source1.tsv"), sep="\t", dtype=str), "Train S1")
train_s2 = preprocess_df(pd.read_csv(find_file("train_source2.tsv"), sep="\t", dtype=str), "Train S2")
train_s3 = preprocess_df(pd.read_csv(find_file("train_source3.tsv"), sep="\t", dtype=str), "Train S3")
train_gt = pd.read_csv(find_file("train_ground_truth.tsv"), sep="\t", dtype=str)

print(f"✅ Training data loaded & preprocessed in {fmt_time(time.time()-t_load)}")
print(f"   Train S1: {len(train_s1):,} | Train S2: {len(train_s2):,} | Train S3: {len(train_s3):,}")




Loading training datasets...
  Preprocessing Train S1 (2,206,821 records)... done in 100.2s
  Preprocessing Train S2 (5,034,616 records)... done in 233.8s
  Preprocessing Train S3 (5,285,603 records)... done in 242.5s
✅ Training data loaded & preprocessed in 10m 12s
   Train S1: 2,206,821 | Train S2: 5,034,616 | Train S3: 5,285,603


In [8]:

# ==============================================================================
# === CELL 4: Ground Truth Parsing (Fast Vectorized Lookup)
# ==============================================================================

print("\nBuilding Ground Truth dictionary...")
gt_dict = {}
gt_s1_ids = train_gt['source1_entity_id'].values
gt_matched = train_gt['matched_entity_ids'].values

for i in range(len(gt_s1_ids)):
    m = gt_matched[i]
    if pd.isna(m) or str(m).strip() == "":
        gt_dict[gt_s1_ids[i]] = set()
    else:
        gt_dict[gt_s1_ids[i]] = set(str(m).split(','))

total_gt_s1 = len(gt_dict)
singletons = sum(1 for v in gt_dict.values() if len(v) == 0)
total_matches = sum(len(v) for v in gt_dict.values())
avg_matches = total_matches / max(total_gt_s1 - singletons, 1)

print(f"  Total S1 Entities: {total_gt_s1:,}")
print(f"  Singletons (0 matches): {singletons:,} ({singletons / total_gt_s1 * 100:.2f}%)")
print(f"  Entities with Matches: {total_gt_s1 - singletons:,}")
print(f"  Total Ground Truth Matches: {total_matches:,} (Avg {avg_matches:.2f} matches/non-singleton)")

# Free raw train_gt dataframe — gt_dict is already constructed!
del train_gt, gt_s1_ids, gt_matched
gc.collect()




Building Ground Truth dictionary...
  Total S1 Entities: 2,206,821
  Singletons (0 matches): 123,247 (5.58%)
  Entities with Matches: 2,083,574
  Total Ground Truth Matches: 7,638,365 (Avg 3.67 matches/non-singleton)


0

In [9]:

# ==============================================================================
# === CELL 5: Multilingual Dense Embedding Engine (GPU-Accelerated)
# ==============================================================================
# WHY MULTILINGUAL EMBEDDINGS ARE MANDATORY:
# - Indian records in S2/S3 frequently feature Hindi (Devanagari) & Tamil scripts.
# - Test set contains France (unseen in training).
# - 'paraphrase-multilingual-MiniLM-L12-v2' maps all these scripts into a unified
#   384-dimensional semantic space, allowing cross-lingual matching.
# ==============================================================================

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DEVICE = 'cuda:0' if CUDA_AVAILABLE else 'cpu'

print(f"\nInitializing Multilingual Embedding Model ({MODEL_NAME}) on {EMBED_DEVICE}...")
t_emb_init = time.time()
embed_model = SentenceTransformer(MODEL_NAME, device=EMBED_DEVICE)
print(f"✅ Model ready ({time.time()-t_emb_init:.1f}s). Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


def encode_texts_to_disk(texts, output_path, desc="texts", batch_size=256, max_len=200):
    """
    Encode strings into L2-normalized float32 vectors directly streamed to disk (.npy).
    OOM-IMMUNE: Uses np.lib.format.open_memmap so the 15.9 GB matrix is NEVER held in RAM.
    Peak memory stays around ~500 MB (one chunk at a time).
    RESUMABLE: Saves chunk progress to disk; if interrupted, resumes from last completed chunk.
    """
    n_total = len(texts)
    _dim = embed_model.get_sentence_embedding_dimension()
    t0 = time.time()
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    
    progress_file = output_path + ".progress"
    CHUNK_SIZE = 500000
    n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE
    
    # Check if already 100% completed
    if os.path.exists(output_path) and not os.path.exists(progress_file):
        try:
            cached = np.load(output_path, mmap_mode='r')
            if cached.shape == (n_total, _dim):
                print(f"  ✅ [Cache hit] {desc} already completed: {output_path} (shape {cached.shape})", flush=True)
                return cached
        except Exception:
            pass

    # Check if resuming from partial progress
    start_chunk = 0
    if os.path.exists(output_path) and os.path.exists(progress_file):
        try:
            with open(progress_file, 'r') as f:
                start_chunk = int(f.read().strip())
            mmap_arr = np.lib.format.open_memmap(output_path, mode='r+')
            if mmap_arr.shape == (n_total, _dim) and start_chunk < n_chunks:
                print(f"  🔄 Resuming {desc} from Chunk {start_chunk + 1}/{n_chunks} (saved progress detected!)...", flush=True)
            else:
                start_chunk = 0
                mmap_arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float32, shape=(n_total, _dim))
        except Exception as e:
            print(f"  ⚠️ Could not resume ({e}), starting fresh...", flush=True)
            start_chunk = 0
            mmap_arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float32, shape=(n_total, _dim))
    else:
        mmap_arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float32, shape=(n_total, _dim))
        with open(progress_file, 'w') as f:
            f.write('0')

    if GPU_COUNT >= 2:
        target_devices = [f'cuda:{i}' for i in range(GPU_COUNT)]
        print(f"  Encoding {desc}: {n_total:,} items | {GPU_COUNT} GPUs {target_devices} | "
              f"{n_chunks} chunks | streaming to disk ({n_total*_dim*4/1e9:.1f} GB)", flush=True)
        try:
            pool = embed_model.start_multi_process_pool(target_devices=target_devices)
            for ci in range(start_chunk, n_chunks):
                c_start = ci * CHUNK_SIZE
                c_end = min(c_start + CHUNK_SIZE, n_total)
                t_chunk = time.time()
                
                # Truncate strings ONLY for this chunk — never hold 10M Python strings in RAM!
                chunk_raw = texts[c_start:c_end]
                if hasattr(chunk_raw, 'tolist'):
                    chunk_raw = chunk_raw.tolist()
                chunk_texts = [t[:max_len] if len(t) > max_len else t for t in chunk_raw]
                del chunk_raw
                
                chunk_emb = embed_model.encode_multi_process(
                    chunk_texts, pool, batch_size=batch_size
                )
                norms = np.linalg.norm(chunk_emb, axis=1, keepdims=True)
                mmap_arr[c_start:c_end] = (chunk_emb / np.maximum(norms, 1e-8)).astype(np.float32)
                mmap_arr.flush()
                
                with open(progress_file, 'w') as f:
                    f.write(str(ci + 1))
                    
                del chunk_texts, chunk_emb, norms
                gc.collect()
                
                elapsed = time.time() - t0
                rate = c_end / max(elapsed, 0.001)
                eta = (n_total - c_end) / max(rate, 1)
                pct = c_end / n_total * 100
                print(f"    [{desc}] Chunk {ci+1:2d}/{n_chunks}: {c_end:9,}/{n_total:,} ({pct:5.1f}%) | "
                      f"{rate:5.0f}/s | Chunk: {time.time()-t_chunk:.0f}s | ETA: {fmt_time(eta)}", flush=True)
                      
            embed_model.stop_multi_process_pool(pool)
            del mmap_arr
            gc.collect()
            if os.path.exists(progress_file):
                os.remove(progress_file)
            elapsed = time.time() - t0
            print(f"  ✅ {desc} done (dual-GPU): shape ({n_total}, {_dim}) in {fmt_time(elapsed)} | {n_total/max(elapsed,0.001):.0f}/s")
            return np.load(output_path, mmap_mode='r')
        except Exception as e:
            print(f"  ⚠️ Dual-GPU pool failed ({e}), using single-GPU fallback...", flush=True)

    # Single GPU / CPU fallback — also streamed directly to disk
    print(f"  Encoding {desc}: {n_total:,} items (device={EMBED_DEVICE}, batch={batch_size})...", flush=True)
    for ci in range(start_chunk, n_chunks):
        c_start = ci * CHUNK_SIZE
        c_end = min(c_start + CHUNK_SIZE, n_total)
        t_chunk = time.time()
        
        chunk_raw = texts[c_start:c_end]
        if hasattr(chunk_raw, 'tolist'):
            chunk_raw = chunk_raw.tolist()
        chunk_texts = [t[:max_len] if len(t) > max_len else t for t in chunk_raw]
        del chunk_raw
        
        chunk_emb = embed_model.encode(
            chunk_texts, batch_size=batch_size,
            show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True
        )
        mmap_arr[c_start:c_end] = chunk_emb.astype(np.float32)
        mmap_arr.flush()
        
        with open(progress_file, 'w') as f:
            f.write(str(ci + 1))
            
        del chunk_texts, chunk_emb
        gc.collect()
        
        elapsed = time.time() - t0
        rate = c_end / max(elapsed, 0.001)
        eta = (n_total - c_end) / max(rate, 1)
        pct = c_end / n_total * 100
        print(f"    [{desc}] Chunk {ci+1:2d}/{n_chunks}: {c_end:9,}/{n_total:,} ({pct:5.1f}%) | "
              f"{rate:5.0f}/s | Chunk: {time.time()-t_chunk:.0f}s | ETA: {fmt_time(eta)}", flush=True)
              
    del mmap_arr
    gc.collect()
    if os.path.exists(progress_file):
        os.remove(progress_file)
    print(f"  ✅ {desc} done: shape ({n_total}, {_dim}) in {fmt_time(time.time()-t0)}")
    return np.load(output_path, mmap_mode='r')


def encode_texts_in_batches(texts, desc="texts", batch_size=256, max_len=200):
    """Memory-safe wrapper for legacy callers (< 1M items)."""
    tmp_path = os.path.join(OUTPUT_DIR, f"_temp_{desc.lower().replace(' ', '_')}.npy")
    res = encode_texts_to_disk(texts, tmp_path, desc=desc, batch_size=batch_size, max_len=max_len)
    arr = np.array(res, copy=True)
    del res
    gc.collect()
    try:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
    except:
        pass
    return arr



Initializing Multilingual Embedding Model (paraphrase-multilingual-MiniLM-L12-v2) on cuda:0...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model ready (15.3s). Embedding dimension: 384


In [10]:


# ==============================================================================
# === CELL 6: Training Data Subsampling Strategy
# ==============================================================================
# SELECTION: 500,000 Source 1 Entities (~22.7% of full S1 training data).
# - On AWS g5.4xlarge, 500K entities generates ~1.7M positive pairs and ~15M negative pairs.
# - Statistical convergence for LightGBM is achieved at 1M+ pairs; 500K provides ample
#   diversity while completing feature extraction in ~35-45 minutes.
# - Searches across 100% of Source 2 (5.03M) and Source 3 (5.29M) records.
# ==============================================================================

# ------------------------------------------------------------------------------
# Training Scale Configuration:
# - Set USE_100_PERCENT_TRAIN = True to use 100% of Source 1 training data (~2.2M entities)
#   Split in Cell 9: 80% Train (~1.76M S1 entities) / 20% Validation (~440k S1 entities).
#   Runtime: ~6.5 - 7.5 hours on Kaggle dual T4 GPU (safely within the 9-hour limit).
# - Set USE_100_PERCENT_TRAIN = False to use a balanced subsample (300k on Kaggle, 500k on AWS)
#   Runtime: ~3.5 - 4.0 hours (leaves ~5 hours buffer for rapid iteration).
# ------------------------------------------------------------------------------
USE_100_PERCENT_TRAIN = True

TRAIN_S1_SAMPLE_SIZE = 300000 if IS_KAGGLE else 500000

np.random.seed(42)
all_s1_ids = train_s1['entity_id'].unique()
if USE_100_PERCENT_TRAIN or len(all_s1_ids) <= TRAIN_S1_SAMPLE_SIZE:
    train_sub_s1 = train_s1
    sampled_s1_id_set = set(all_s1_ids)
    print(f"\n🚀 USING 100% FULL TRAINING S1 DATA: {len(train_sub_s1):,} entities!")
    print(f"   Cell 9 split: 80% Train (~{int(len(train_sub_s1)*0.8):,} S1) / 20% Val (~{int(len(train_sub_s1)*0.2):,} S1)")
else:
    sampled_s1_id_set = set(np.random.choice(all_s1_ids, TRAIN_S1_SAMPLE_SIZE, replace=False))
    train_sub_s1 = train_s1[train_s1['entity_id'].isin(sampled_s1_id_set)].reset_index(drop=True)
    print(f"\nSubsampled Training S1: {len(train_sub_s1):,} / {len(train_s1):,} ({len(train_sub_s1)/len(train_s1)*100:.1f}%)")
    print(f"   Cell 9 split: 80% Train (~{int(len(train_sub_s1)*0.8):,} S1) / 20% Val (~{int(len(train_sub_s1)*0.2):,} S1)")

# CRITICAL MEMORY OPTIMIZATION: Free full train_s1 (2.2M rows, ~2GB RAM)
del train_s1, all_s1_ids
gc.collect()

train_sub_gt = {k: v for k, v in gt_dict.items() if k in sampled_s1_id_set}
train_s2s3 = pd.concat([train_s2, train_s3], ignore_index=True)
print(f"Combined Training S2+S3 Target Universe: {len(train_s2s3):,} records (100% included)")

# CRITICAL MEMORY OPTIMIZATION: Free train_s2 and train_s3 (10.3M rows combined, ~8GB RAM)
del train_s2, train_s3
gc.collect()

# Generate or load cached training embeddings
sub_s1_emb_path = os.path.join(OUTPUT_DIR, "train_sub_s1_emb.npy")
train_s2s3_emb_path = os.path.join(OUTPUT_DIR, "train_s2s3_emb.npy")
emb_batch = 256 if IS_KAGGLE else 512

if os.path.exists(sub_s1_emb_path) and not os.path.exists(sub_s1_emb_path + ".progress"):
    try:
        sub_s1_embeddings = np.load(sub_s1_emb_path)
        if len(sub_s1_embeddings) == len(train_sub_s1):
            print(f"Loading cached {sub_s1_emb_path} ({len(sub_s1_embeddings):,} vectors)...")
        else:
            print(f"⚠️ Cached {sub_s1_emb_path} size ({len(sub_s1_embeddings):,}) does not match current S1 ({len(train_sub_s1):,}). Re-encoding...")
            os.remove(sub_s1_emb_path)
            sub_s1_embeddings = encode_texts_to_disk(train_sub_s1['embed_text'], sub_s1_emb_path, desc="Train S1", batch_size=emb_batch)
    except Exception:
        sub_s1_embeddings = encode_texts_to_disk(train_sub_s1['embed_text'], sub_s1_emb_path, desc="Train S1", batch_size=emb_batch)
else:
    sub_s1_embeddings = encode_texts_to_disk(train_sub_s1['embed_text'], sub_s1_emb_path, desc="Train S1", batch_size=emb_batch)

train_s2s3_progress = train_s2s3_emb_path + ".progress"
if os.path.exists(train_s2s3_emb_path) and not os.path.exists(train_s2s3_progress):
    print(f"Loading cached {train_s2s3_emb_path} (mmap_mode='r')...")
    train_s2s3_embeddings = np.load(train_s2s3_emb_path, mmap_mode='r')
    assert len(train_s2s3_embeddings) == len(train_s2s3), "Cached S2S3 embedding size mismatch! Delete cache file and retry."
else:
    train_s2s3_embeddings = encode_texts_to_disk(train_s2s3['embed_text'], train_s2s3_emb_path, desc="Train S2+S3", batch_size=emb_batch)




🚀 USING 100% FULL TRAINING S1 DATA: 2,206,821 entities!
   Cell 9 split: 80% Train (~1,765,456 S1) / 20% Val (~441,364 S1)
Combined Training S2+S3 Target Universe: 10,320,219 records (100% included)
  Encoding Train S1: 2,206,821 items | 2 GPUs ['cuda:0', 'cuda:1'] | 5 chunks | streaming to disk (3.4 GB)
    [Train S1] Chunk  1/5:   500,000/2,206,821 ( 22.7%) |  2835/s | Chunk: 154s | ETA: 10m 02s
    [Train S1] Chunk  2/5: 1,000,000/2,206,821 ( 45.3%) |  2983/s | Chunk: 159s | ETA: 6m 44s
    [Train S1] Chunk  3/5: 1,500,000/2,206,821 ( 68.0%) |  3039/s | Chunk: 158s | ETA: 3m 52s
    [Train S1] Chunk  4/5: 2,000,000/2,206,821 ( 90.6%) |  3066/s | Chunk: 159s | ETA: 1m 07s
    [Train S1] Chunk  5/5: 2,206,821/2,206,821 (100.0%) |  3071/s | Chunk: 66s | ETA: 0s
  ✅ Train S1 done (dual-GPU): shape (2206821, 384) in 12m 00s | 3063/s
  Encoding Train S2+S3: 10,320,219 items | 2 GPUs ['cuda:0', 'cuda:1'] | 21 chunks | streaming to disk (15.9 GB)
    [Train S2+S3] Chunk  1/21:   500,000/10

In [ ]:

# ==============================================================================
# === CELL 7: Scalable Candidate Generation (Dual Dense + Sparse Blocking)
# ==============================================================================
# AMAZON EVALUATION RULE:
# "The approach that generates a smaller candidate set per Source 1 entity will be
# ranked higher in the final evaluation beyond the public/private leaderboard."
#
# DESIGN:
# - S1 is the QUERY (180K/500K queries against 10M chunked targets).
# - Strategy A: Multilingual dense embedding ANN via FAISS GPU (top-8 per S1, thr >= 0.35)
# - Strategy B: TF-IDF char 2-4 ngrams on Name (top-8 per S1, thr >= 0.15)
# - Strategy C: TF-IDF char 2-4 ngrams on Name+Address (top-5 per S1, thr >= 0.20)
# - Global Union & Deduplication: Keep strictly TOP-8 to TOP-10 candidates per S1 entity!
# - Result: >96% blocking recall with an average of only ~6 to 8 candidates per entity!
# ==============================================================================

def sparse_topn(A, B_T, top_n, threshold, n_threads=NUM_CPUS):
    """Platform-independent wrapper for high-speed sparse top-N multiplication."""
    if USE_NEW_API:
        return sp_matmul_topn(A, B_T, top_n=top_n, threshold=threshold, n_threads=n_threads)
    else:
        return awesome_cossim_topn(A, B_T, ntop=top_n, lower_bound=threshold,
                                   use_threads=True, n_jobs=n_threads)


def tfidf_query_blocking(s1_df, s2s3_df, field='name_norm', top_k=8, threshold=0.15):
    """
    High-speed chunked TF-IDF blocking where S1 is the QUERY.
    Fits vocabulary on S1 only; transforms S2/S3 in 500k chunks.
    """
    n_s1 = len(s1_df)
    n_s2s3 = len(s2s3_df)
    if n_s1 == 0 or n_s2s3 == 0:
        return []
    
    t0 = time.time()
    s1_texts = s1_df[field].fillna('').tolist()
    s2s3_texts = s2s3_df[field].fillna('').tolist()
    s1_ids = s1_df['entity_id'].values
    s2s3_ids = s2s3_df['entity_id'].values
    
    # 1. Fit TF-IDF on S1 queries (char 3-5 n-grams with min_df=2 filters common noise)
    vectorizer = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 5), min_df=2,
        max_features=150000, dtype=np.float32, sublinear_tf=True
    )
    print(f"      [{field}] Fitting TF-IDF on {n_s1:,} S1 queries...", end=" ", flush=True)
    s1_tfidf = vectorizer.fit_transform(s1_texts)
    print(f"done. Vocab size: {len(vectorizer.vocabulary_):,}")
    
    chunk_size = 500000
    n_chunks = (n_s2s3 + chunk_size - 1) // chunk_size
    print(f"      [{field}] Querying {n_s2s3:,} S2+S3 records in {n_chunks} chunks...")
    acc_s1, acc_s2s3, acc_score = [], [], []
    
    for ci in range(n_chunks):
        start = ci * chunk_size
        end = min(start + chunk_size, n_s2s3)
        chunk_tfidf = vectorizer.transform(s2s3_texts[start:end])
        
        # KEY ARCHITECTURE: S1 is the first argument (Row queries)
        cos_matrix = sparse_topn(s1_tfidf, chunk_tfidf.T, top_n=top_k, threshold=threshold, n_threads=NUM_CPUS)
        coo = cos_matrix.tocoo()
        
        if len(coo.data) > 0:
            acc_s1.append(coo.row.astype(np.int32))
            acc_s2s3.append((coo.col + start).astype(np.int32))
            acc_score.append(coo.data.astype(np.float32))
        
        # Per-chunk progress print  
        elapsed = time.time() - t0
        rate = end / max(elapsed, 0.001)
        eta = (n_s2s3 - end) / max(rate, 1)
        pct = end / n_s2s3 * 100
        print(f"        Chunk {ci+1:2d}/{n_chunks}: {end:9,}/{n_s2s3:,} ({pct:5.1f}%) | {rate/1000:.1f}K rows/s | ETA: {fmt_time(eta)} | pairs so far: {sum(len(x) for x in acc_s1):,}", flush=True)
            
        del chunk_tfidf, cos_matrix, coo
        gc.collect()
        
    del s1_tfidf
    gc.collect()
    
    if not acc_s1:
        return []
    
    cat_s1 = np.concatenate(acc_s1)
    cat_s2s3 = np.concatenate(acc_s2s3)
    cat_score = np.concatenate(acc_score)
    
    # Merge and take global top-K per S1 entity
    if n_chunks > 1:
        df_merge = pd.DataFrame({'s1': cat_s1, 's2s3': cat_s2s3, 'sc': cat_score})
        df_merge = df_merge.sort_values('sc', ascending=False).groupby('s1').head(top_k).reset_index(drop=True)
        res_s1 = df_merge['s1'].values
        res_s2s3 = df_merge['s2s3'].values
        res_sc = df_merge['sc'].values
        del df_merge
    else:
        res_s1, res_s2s3, res_sc = cat_s1, cat_s2s3, cat_score
        
    candidates = list(zip(s1_ids[res_s1], s2s3_ids[res_s2s3], res_sc.astype(float)))
    print(f"      [{field}] → {len(candidates):,} pairs in {fmt_time(time.time()-t0)}")
    return candidates


def faiss_gpu_blocking(s1_df, s1_emb, s2s3_df, s2s3_emb, top_k=8, threshold=0.35):
    """
    Multilingual semantic nearest-neighbor search using FAISS GPU/CPU.
    For large S2/S3 corpora (>4M), adds S2/S3 in chunks to avoid GPU VRAM OOM.
    Queries S1 in batches of 20K for progress tracking.
    """
    n_s1 = len(s1_df)
    n_s2s3 = len(s2s3_df)
    if n_s1 == 0 or n_s2s3 == 0:
        return []
    
    t0 = time.time()
    s1_ids = s1_df['entity_id'].values
    s2s3_ids = s2s3_df['entity_id'].values
    dim = s1_emb.shape[1]
    
    # VRAM & RAM budget: 2×T4 = 2×15GB = 30GB total.
    # Safe chunk size: 1.5M vectors × 384 × 4B = 2.3 GB per chunk — safe within 30 GB RAM & 15 GB VRAM.
    MAX_INDEX_SIZE = 1500000
    use_gpu = FAISS_GPUS > 0 and CUDA_AVAILABLE

    def _build_index(emb_chunk):
        """Build a FAISS flat IP index, replicated across all available GPUs."""
        idx_cpu = faiss.IndexFlatIP(dim)
        idx_cpu.add(emb_chunk.astype(np.float32))
        if use_gpu:
            try:
                if GPU_COUNT >= 2:
                    # Replicate index across BOTH GPUs — search is parallelized automatically
                    co = faiss.GpuMultipleClonerOptions()
                    co.shard = False   # replicate (not shard): full index on each GPU
                    co.useFloat16 = True  # float16 halves VRAM, negligible accuracy loss
                    gpu_idx = faiss.index_cpu_to_all_gpus(idx_cpu, co=co)
                    del idx_cpu  # Free CPU RAM copy immediately after cloning to GPUs!
                    gc.collect()
                    return gpu_idx
                else:
                    res = faiss.StandardGpuResources()
                    gpu_idx = faiss.index_cpu_to_gpu(res, 0, idx_cpu)
                    del idx_cpu
                    gc.collect()
                    return gpu_idx
            except Exception as e:
                print(f"  (FAISS GPU failed, CPU fallback: {e})", flush=True)
        faiss.omp_set_num_threads(NUM_CPUS)
        return idx_cpu

    # Query in batches of 20,000 S1 entities
    q_batch_size = 20000
    n_q_batches = (n_s1 + q_batch_size - 1) // q_batch_size
    candidates = []

    # Process S2S3 in chunks if too large for GPU VRAM
    s2s3_chunk_size = MAX_INDEX_SIZE if n_s2s3 > MAX_INDEX_SIZE else n_s2s3
    n_s2s3_chunks = (n_s2s3 + s2s3_chunk_size - 1) // s2s3_chunk_size
    
    for s2s3_ci in range(n_s2s3_chunks):
        s2s3_start = s2s3_ci * s2s3_chunk_size
        s2s3_end = min(s2s3_start + s2s3_chunk_size, n_s2s3)
        s2s3_ids_chunk = s2s3_ids[s2s3_start:s2s3_end]
        s2s3_emb_chunk = s2s3_emb[s2s3_start:s2s3_end]

        index = _build_index(s2s3_emb_chunk)
        del s2s3_emb_chunk  # free immediately after indexing

        report_qb_every = max(1, n_q_batches // 5)  # report every 20% of query batches
        for qb in range(n_q_batches):
            q_start = qb * q_batch_size
            q_end = min(q_start + q_batch_size, n_s1)
            q_vecs = s1_emb[q_start:q_end]

            scores, nn_indices = index.search(q_vecs, top_k)

            for i in range(len(q_vecs)):
                s1_global_idx = q_start + i
                for k in range(scores.shape[1]):
                    sim = float(scores[i, k])
                    t_idx = int(nn_indices[i, k])
                    if t_idx >= 0 and sim >= threshold:
                        candidates.append((s1_ids[s1_global_idx], s2s3_ids_chunk[t_idx], sim))
            
            if (qb + 1) % report_qb_every == 0 or (qb + 1) == n_q_batches:
                elapsed_q = time.time() - t0
                pct_q = q_end / n_s1 * 100
                print(f"        [FAISS S2S3 {s2s3_ci+1}/{n_s2s3_chunks}] Queries: {q_end:,}/{n_s1:,} ({pct_q:.0f}%) | {len(candidates):,} pairs", flush=True)

        del index
        gc.collect()
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()
        if n_s2s3_chunks > 1:
            pct = s2s3_end / n_s2s3 * 100
            print(f"      [FAISS] S2S3 chunk {s2s3_ci+1}/{n_s2s3_chunks} ({pct:.0f}%) → {len(candidates):,} raw pairs so far")

    print(f"      [FAISS Dense] → {len(candidates):,} pairs in {fmt_time(time.time()-t0)}")
    return candidates


def generate_candidate_universe(s1_df, *args, **kwargs):
    """
    Executes country-partitioned multi-channel candidate generation and enforces
    strict top-K pruning per S1 entity to maximize candidate efficiency.
    Supports both clean signature:
      (s1_df, s1_emb, s2s3_df, s2s3_emb, max_candidates_per_s1=10)
    and legacy signature:
      (s1_df, s2_df, s3_df, s1_emb, s2s3_df, s2s3_emb, max_candidates_per_s1=10)
    """
    if len(args) >= 5:
        s1_emb = args[2]
        s2s3_df = args[3]
        s2s3_emb = args[4]
    elif len(args) >= 3:
        s1_emb = args[0]
        s2s3_df = args[1]
        s2s3_emb = args[2]
    else:
        s1_emb = kwargs['s1_emb']
        s2s3_df = kwargs['s2s3_df']
        s2s3_emb = kwargs['s2s3_emb']
    max_candidates_per_s1 = kwargs.get('max_candidates_per_s1', 10)

    t_start = time.time()
    countries = s1_df['country'].dropna().unique()
    print(f"  Partitioning blocking across countries: {list(countries)}")
    
    country_dfs = []
    
    for c_idx, country in enumerate(countries, 1):
        s1_mask = s1_df['country'].values == country
        s2s3_mask = s2s3_df['country'].values == country
        
        s1_c = s1_df[s1_mask].reset_index(drop=True)
        s2s3_c = s2s3_df[s2s3_mask].reset_index(drop=True)
        s1_emb_c = s1_emb[s1_mask]
        s2s3_emb_c = s2s3_emb[s2s3_mask]
        
        print(f"\n  🔍 [{c_idx}/{len(countries)}] Country: {country} ({len(s1_c):,} S1 queries × {len(s2s3_c):,} S2/S3 targets)")
        
        # Channel 1: Dense Multilingual Embedding NN
        cands_emb = faiss_gpu_blocking(s1_c, s1_emb_c, s2s3_c, s2s3_emb_c, top_k=8, threshold=0.35)
        
        # Channel 2: TF-IDF char n-grams on Name
        cands_name = tfidf_query_blocking(s1_c, s2s3_c, field='name_norm', top_k=8, threshold=0.25)
        
        # Channel 3: TF-IDF char n-grams on Name+Address
        cands_addr = tfidf_query_blocking(s1_c, s2s3_c, field='name_addr', top_k=5, threshold=0.25)
        
        # Free country input tensors immediately
        del s1_c, s2s3_c, s1_emb_c, s2s3_emb_c
        gc.collect()
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()

        # Deduplicate & prune THIS country immediately to keep RAM ultra-low (< 300 MB)
        c_raw = cands_emb + cands_name + cands_addr
        del cands_emb, cands_name, cands_addr
        
        if c_raw:
            c_df = pd.DataFrame(c_raw, columns=['s1_id', 's2s3_id', 'block_score'])
            del c_raw
            # Deduplicate per pair: keep max score across channels
            c_df = c_df.groupby(['s1_id', 's2s3_id'], as_index=False)['block_score'].max()
            # Compact pruning: Keep top-N highest scoring candidate pairs per S1 entity
            c_df = c_df.sort_values(['s1_id', 'block_score'], ascending=[True, False]).groupby('s1_id').head(max_candidates_per_s1).reset_index(drop=True)
            country_dfs.append(c_df)
            del c_df
            gc.collect()

    t_dedup = time.time()
    cands_df = pd.concat(country_dfs, ignore_index=True) if country_dfs else pd.DataFrame(columns=['s1_id', 's2s3_id', 'block_score'])
    del country_dfs
    gc.collect()
    cands_df = cands_df.rename(columns={'block_score': 'tfidf_cosine'})
    
    avg_per_s1 = len(cands_df) / max(len(s1_df), 1)
    print(f"  ✅ Pruning complete in {time.time()-t_dedup:.1f}s")
    print(f"     Total Unique Candidate Pairs: {len(cands_df):,}")
    print(f"     Average Candidates per S1 Entity: {avg_per_s1:.2f} (Target: <= 10.0 for max bonus!)")
    print(f"     Total Blocking Time: {fmt_time(time.time()-t_start)}")
    return cands_df


print("\n" + "=" * 80)
print("BLOCKING ON TRAINING SUBSET")
print("=" * 80)

train_cand_path = os.path.join(OUTPUT_DIR, "train_candidates_v4.csv")
if os.path.exists(train_cand_path):
    print(f"Loading cached {train_cand_path}...")
    train_candidates = pd.read_csv(train_cand_path)
else:
    train_candidates = generate_candidate_universe(
        train_sub_s1, sub_s1_embeddings, train_s2s3, train_s2s3_embeddings,
        max_candidates_per_s1=10
    )
    train_candidates.to_csv(train_cand_path, index=False)
    print(f"💾 Checkpoint saved: {train_cand_path}")

# Evaluate blocking recall on the training subset
cand_set = set(zip(train_candidates['s1_id'], train_candidates['s2s3_id']))
total_true_links = sum(len(v) for v in train_sub_gt.values())
found_true_links = sum(1 for s1, mids in train_sub_gt.items() for mid in mids if (s1, mid) in cand_set)
block_recall = found_true_links / total_true_links if total_true_links > 0 else 0

print(f"\n🎯 TRAINING CANDIDATE RECALL: {block_recall:.4f} ({found_true_links:,} / {total_true_links:,})")
print(f"   Candidate set size: {len(train_candidates):,} pairs across {len(train_sub_s1):,} queries.")



BLOCKING ON TRAINING SUBSET
  Partitioning blocking across countries: ['US', 'India']

  🔍 [1/2] Country: US (1,323,633 S1 queries × 6,186,873 S2/S3 targets)
        [FAISS S2S3 1/5] Queries: 260,000/1,323,633 (20%) | 2,080,000 pairs
        [FAISS S2S3 1/5] Queries: 520,000/1,323,633 (39%) | 4,160,000 pairs
        [FAISS S2S3 1/5] Queries: 780,000/1,323,633 (59%) | 6,240,000 pairs
        [FAISS S2S3 1/5] Queries: 1,040,000/1,323,633 (79%) | 8,320,000 pairs
        [FAISS S2S3 1/5] Queries: 1,300,000/1,323,633 (98%) | 10,400,000 pairs
        [FAISS S2S3 1/5] Queries: 1,323,633/1,323,633 (100%) | 10,589,064 pairs
      [FAISS] S2S3 chunk 1/5 (24%) → 10,589,064 raw pairs so far
        [FAISS S2S3 2/5] Queries: 260,000/1,323,633 (20%) | 12,669,064 pairs
        [FAISS S2S3 2/5] Queries: 520,000/1,323,633 (39%) | 14,749,064 pairs
        [FAISS S2S3 2/5] Queries: 780,000/1,323,633 (59%) | 16,829,064 pairs
        [FAISS S2S3 2/5] Queries: 1,040,000/1,323,633 (79%) | 18,909,064 pairs
 

In [ ]:


# ==============================================================================
# === CELL 8: Vectorized 18-Feature Engineering Engine
# ==============================================================================

def precompute_embedding_index(df):
    """Maps entity ID to row index for O(1) embedding lookup."""
    ids = df['entity_id'].values
    return {ids[i]: i for i in range(len(ids))}


sub_s1_emb_map = precompute_embedding_index(train_sub_s1)
train_s2s3_emb_map = precompute_embedding_index(train_s2s3)


def compute_feature_batch(s1_names, s2_names, s1_addrs, s2_addrs,
                          s1_ids, s2_ids, tfidf_scores,
                          s1_emb, s2_emb, s1_map, s2_map):
    """Computes all 18 discriminative features for a batch of candidate pairs."""
    n = len(s1_names)
    feats = {
        'tfidf_cosine': np.array(tfidf_scores, dtype=np.float32),
        'embed_cosine': np.zeros(n, dtype=np.float32),
        'name_fuzz_ratio': np.zeros(n, dtype=np.float32),
        'name_partial': np.zeros(n, dtype=np.float32),
        'name_token_sort': np.zeros(n, dtype=np.float32),
        'name_token_set': np.zeros(n, dtype=np.float32),
        'name_jaro_winkler': np.zeros(n, dtype=np.float32),
        'name_jaccard_3gram': np.zeros(n, dtype=np.float32),
        'name_token_jaccard': np.zeros(n, dtype=np.float32),
        'name_len_ratio': np.zeros(n, dtype=np.float32),
        'name_len_diff': np.zeros(n, dtype=np.float32),
        'addr_fuzz_ratio': np.zeros(n, dtype=np.float32),
        'addr_token_sort': np.zeros(n, dtype=np.float32),
        'addr_token_set': np.zeros(n, dtype=np.float32),
        'addr_token_jaccard': np.zeros(n, dtype=np.float32),
        'addr_numeric_jaccard': np.zeros(n, dtype=np.float32),
        'addr_missing': np.zeros(n, dtype=np.float32),
        'source_is_s3': np.zeros(n, dtype=np.float32),
        'combined_score': np.zeros(n, dtype=np.float32),
        'high_conf_name': np.zeros(n, dtype=np.float32),
    }

    # Vectorized dense embedding cosine — single batch mmap read, ~100x faster than per-pair loop
    e1_idxs = np.array([s1_map.get(sid, -1) for sid in s1_ids])
    e2_idxs = np.array([s2_map.get(sid, -1) for sid in s2_ids])
    valid_embs = (e1_idxs >= 0) & (e2_idxs >= 0)
    if np.any(valid_embs):
        v1 = s1_emb[e1_idxs[valid_embs]]
        v2 = s2_emb[e2_idxs[valid_embs]]
        feats['embed_cosine'][valid_embs] = np.sum(v1 * v2, axis=1).astype(np.float32)
    
    for i in range(n):
        n1, n2 = s1_names[i], s2_names[i]
        a1, a2 = s1_addrs[i], s2_addrs[i]
        
        # Name features
        if n1 and n2:
            feats['name_fuzz_ratio'][i] = fuzz.ratio(n1, n2) / 100.0
            feats['name_partial'][i] = fuzz.partial_ratio(n1, n2) / 100.0
            feats['name_token_sort'][i] = fuzz.token_sort_ratio(n1, n2) / 100.0
            feats['name_token_set'][i] = fuzz.token_set_ratio(n1, n2) / 100.0
            feats['name_jaro_winkler'][i] = JaroWinkler.normalized_similarity(n1, n2)
            
            ng1 = set(n1[j:j+3] for j in range(max(0, len(n1)-2)))
            ng2 = set(n2[j:j+3] for j in range(max(0, len(n2)-2)))
            if ng1 or ng2:
                feats['name_jaccard_3gram'][i] = len(ng1 & ng2) / len(ng1 | ng2)
                
            t1, t2 = set(n1.split()), set(n2.split())
            if t1 or t2:
                feats['name_token_jaccard'][i] = len(t1 & t2) / len(t1 | t2)
                
            feats['name_len_ratio'][i] = min(len(n1), len(n2)) / max(len(n1), len(n2))
            feats['name_len_diff'][i] = abs(len(n1) - len(n2))
            if feats['name_fuzz_ratio'][i] > 0.95 or feats['name_token_sort'][i] > 0.95:
                feats['high_conf_name'][i] = 1.0
                
        # Address features
        if a1 and a2:
            feats['addr_fuzz_ratio'][i] = fuzz.ratio(a1, a2) / 100.0
            feats['addr_token_sort'][i] = fuzz.token_sort_ratio(a1, a2) / 100.0
            feats['addr_token_set'][i] = fuzz.token_set_ratio(a1, a2) / 100.0
            at1, at2 = set(a1.split()), set(a2.split())
            if at1 or at2:
                feats['addr_token_jaccard'][i] = len(at1 & at2) / len(at1 | at2)
            num1 = set(re.findall(r'\b\d+\b', a1))
            num2 = set(re.findall(r'\b\d+\b', a2))
            if num1 or num2:
                feats['addr_numeric_jaccard'][i] = len(num1 & num2) / len(num1 | num2)
        else:
            feats['addr_missing'][i] = 1.0
            
        feats['source_is_s3'][i] = 1.0 if s2_ids[i].startswith('S3-') else 0.0
        
        # Composite feature: harmonic mean of name and address similarity
        name_sim = max(feats['name_token_sort'][i], feats['name_jaro_winkler'][i], feats['embed_cosine'][i])
        addr_sim = max(feats['addr_token_sort'][i], feats['addr_fuzz_ratio'][i]) if not feats['addr_missing'][i] else name_sim
        feats['combined_score'][i] = (2 * name_sim * addr_sim) / max(name_sim + addr_sim, 1e-4)
        
    return pd.DataFrame(feats)


def extract_features_pipeline(cands_df, s1_df, s2s3_df, s1_emb, s2s3_emb, s1_map, s2_map, desc="dataset"):
    """Orchestrates batch feature extraction with progress tracking."""
    total_pairs = len(cands_df)
    print(f"\nExtracting features for {total_pairs:,} candidate pairs ({desc})...")
    t0 = time.time()
    
    cand_s1_ids = cands_df['s1_id'].values
    cand_s2_ids = cands_df['s2s3_id'].values
    cand_tfidf = cands_df['tfidf_cosine'].values
    
    # Pre-index text fields into dictionaries (filter to candidates only to avoid 10.3M dict in RAM)
    print("  Pre-resolving name and address string lookups...", end=" ", flush=True)
    needed_s2_ids = set(cand_s2_ids)
    s2s3_sub = s2s3_df[s2s3_df['entity_id'].isin(needed_s2_ids)]
    s1_names = s1_df.set_index('entity_id')['name_norm'].to_dict()
    s1_addrs = s1_df.set_index('entity_id')['addr_norm'].to_dict()
    s2_names = s2s3_sub.set_index('entity_id')['name_norm'].to_dict()
    s2_addrs = s2s3_sub.set_index('entity_id')['addr_norm'].to_dict()
    del s2s3_sub
    gc.collect()
    print("done.")
    
    resolved_s1_names = [s1_names.get(x, '') for x in cand_s1_ids]
    resolved_s2_names = [s2_names.get(x, '') for x in cand_s2_ids]
    resolved_s1_addrs = [s1_addrs.get(x, '') for x in cand_s1_ids]
    resolved_s2_addrs = [s2_addrs.get(x, '') for x in cand_s2_ids]
    
    batch_size = 50000
    n_batches = (total_pairs + batch_size - 1) // batch_size
    all_chunks = []
    
    for b in range(n_batches):
        b_t0 = time.time()
        start = b * batch_size
        end = min(start + batch_size, total_pairs)
        
        batch_df = compute_feature_batch(
            resolved_s1_names[start:end], resolved_s2_names[start:end],
            resolved_s1_addrs[start:end], resolved_s2_addrs[start:end],
            cand_s1_ids[start:end], cand_s2_ids[start:end],
            cand_tfidf[start:end],
            s1_emb, s2s3_emb, s1_map, s2_map
        )
        all_chunks.append(batch_df)
        
        elapsed = time.time() - t0
        rate = end / max(elapsed, 0.001)
        eta = (total_pairs - end) / max(rate, 1)
        pct = end / total_pairs * 100
        print(f"    [{desc}] Batch {b+1:3d}/{n_batches}: {end:8,}/{total_pairs:,} ({pct:5.1f}%) | {rate:5.0f} pairs/s | Took {time.time()-b_t0:.1f}s | ETA: {fmt_time(eta)}")
        
    feats_df = pd.concat(all_chunks, ignore_index=True)
    feats_df['s1_id'] = cand_s1_ids
    feats_df['s2s3_id'] = cand_s2_ids
    print(f"  ✅ Features complete: {feats_df.shape} in {fmt_time(time.time()-t0)}")
    return feats_df


train_features_path = os.path.join(OUTPUT_DIR, "train_features_v4.csv")
if os.path.exists(train_features_path):
    print(f"\nLoading cached {train_features_path}...")
    train_features = pd.read_csv(train_features_path)
else:
    train_features = extract_features_pipeline(
        train_candidates, train_sub_s1, train_s2s3,
        sub_s1_embeddings, train_s2s3_embeddings,
        sub_s1_emb_map, train_s2s3_emb_map, desc="training"
    )
    train_features.to_csv(train_features_path, index=False, chunksize=100000)
    print(f"💾 Checkpoint saved: {train_features_path}")

# Assign ground truth labels with progress tracking
print("\nAssigning ground truth labels to candidate pairs...")
labels = np.zeros(len(train_features), dtype=np.int32)
s1_vec = train_features['s1_id'].values
s2_vec = train_features['s2s3_id'].values
total_label_rows = len(labels)
report_step = max(1, total_label_rows // 10)  # report every 10%
t_lbl = time.time()

for i in range(total_label_rows):
    if s2_vec[i] in train_sub_gt.get(s1_vec[i], set()):
        labels[i] = 1
    if (i + 1) % report_step == 0 or (i + 1) == total_label_rows:
        pct = (i + 1) / total_label_rows * 100
        rate = (i + 1) / max(time.time() - t_lbl, 0.001)
        eta = (total_label_rows - (i + 1)) / max(rate, 1)
        print(f"  Labeling: {i+1:,}/{total_label_rows:,} ({pct:.0f}%) | {rate:.0f} rows/s | ETA: {fmt_time(eta)}")

train_features['label'] = labels
n_pos = int(labels.sum())
n_neg = len(labels) - n_pos
print(f"  Labels: {n_pos:,} Positive | {n_neg:,} Negative (1:{n_neg/max(n_pos,1):.1f} Imbalance)")

# Class balance: Cap negatives at 10:1 ratio to prevent training divergence
MAX_NEG_RATIO = 10
if n_neg / max(n_pos, 1) > MAX_NEG_RATIO:
    target_neg = n_pos * MAX_NEG_RATIO
    print(f"  Subsampling negatives to {MAX_NEG_RATIO}:1 ratio ({target_neg:,} negatives)...")
    pos_mask = train_features['label'] == 1
    neg_sampled_idx = train_features[~pos_mask].sample(n=target_neg, random_state=42).index
    keep_indices = train_features[pos_mask].index.tolist() + neg_sampled_idx.tolist()
    train_features = train_features.loc[keep_indices].reset_index(drop=True)
    print(f"  ✅ Dataset ready for training: {len(train_features):,} pairs ({train_features['label'].sum():,} positives)")



In [ ]:

# ==============================================================================
# === CELL 9: Model Training (LightGBM with Anti-Overfitting & GroupSplit)
# ==============================================================================

id_columns = ['s1_id', 's2s3_id', 'label']
feature_cols = [c for c in train_features.columns if c not in id_columns]
print(f"\nModel Features ({len(feature_cols)} total):")
for idx, col in enumerate(feature_cols, 1):
    print(f"   {idx:2d}. {col}")

X = train_features[feature_cols].values.astype(np.float32)
y = train_features['label'].values
groups = train_features['s1_id'].values

# Split strictly by S1 entity group to prevent leakage
print("\nPerforming GroupShuffleSplit (80% Train / 20% Validation by S1 entity)...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_indices, val_indices = next(gss.split(X, y, groups))

X_train, X_val = X[train_indices], X[val_indices]
y_train, y_val = y[train_indices], y[val_indices]

print(f"  Train Set: {len(X_train):,} pairs ({y_train.sum():,} pos, {len(y_train)-y_train.sum():,} neg)")
print(f"  Val Set:   {len(X_val):,} pairs ({y_val.sum():,} pos, {len(y_val)-y_val.sum():,} neg)")

# Try GPU training first (T4 supports LightGBM CUDA — ~3x faster than CPU)
_lgb_device = 'cpu'  # default
if CUDA_AVAILABLE:
    try:
        _test_params = {'objective': 'binary', 'device': 'gpu', 'verbose': -1, 'num_leaves': 4}
        _test_data = lgb.Dataset(np.random.rand(100, 3).astype(np.float32), label=np.random.randint(0, 2, 100))
        lgb.train(_test_params, _test_data, num_boost_round=2)
        _lgb_device = 'gpu'
        print("  ✅ LightGBM GPU device available — training will use GPU.")
    except Exception as _e:
        print(f"  ⚠️ LightGBM GPU not available ({_e}), using CPU with {NUM_CPUS} threads.")

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'device': _lgb_device,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 500,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'scale_pos_weight': float(len(y_train) - y_train.sum()) / max(float(y_train.sum()), 1.0),
    'verbose': -1,
    'n_jobs': NUM_CPUS,
    'seed': 42
}
print(f"  LightGBM config: device={_lgb_device} | {len(feature_cols)} features | up to 1200 rounds (early_stop=50)")

dtrain = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
dval = lgb.Dataset(X_val, label=y_val, feature_name=feature_cols, reference=dtrain)

print("\nFitting LightGBM model...")
t_train = time.time()
model = lgb.train(
    lgb_params, dtrain, num_boost_round=1200,
    valid_sets=[dtrain, dval], valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True), lgb.log_evaluation(period=50)]
)

print(f"✅ Training completed in {fmt_time(time.time()-t_train)}. Best Iteration: {model.best_iteration}")

# Display feature gain
gains = sorted(zip(feature_cols, model.feature_importance('gain')), key=lambda x: -x[1])
print("\nFeature Importances (by Gain):")
for f_name, gain in gains:
    print(f"  {f_name:25s}: {gain:12.1f}")

model_save_path = os.path.join(OUTPUT_DIR, "lgbm_v4_model.txt")
model.save_model(model_save_path)
print(f"💾 Checkpoint saved: {model_save_path}")


In [ ]:


# ==============================================================================
# === CELL 10: Vectorized Macro F0.5 Threshold Sweep & Post-Processing
# ==============================================================================
# FORMULA: F0.5 = (1.25 * Precision * Recall) / (0.25 * Precision + Recall)
# CRITICAL:
# - Precision matters 2x more than recall!
# - Evaluated across ALL S1 validation entities (including true singletons).
# - Singletons getting 0 predictions = 1.0. Singletons getting false positive = 0.0.
# ==============================================================================

print("\n" + "=" * 80)
print("VECTORIZED MACRO F0.5 THRESHOLD OPTIMIZATION")
print("=" * 80)

val_probs = model.predict(X_val)
val_df = train_features.iloc[val_indices][['s1_id', 's2s3_id']].copy()
val_df['prob'] = val_probs

val_s1_unique = set(groups[val_indices])
val_gt_dict = {sid: train_sub_gt.get(sid, set()) for sid in val_s1_unique}

print(f"Validation Reference Entities: {len(val_gt_dict):,}")
val_singletons = sum(1 for v in val_gt_dict.values() if len(v) == 0)
print(f"  With matches: {len(val_gt_dict)-val_singletons:,} | Singletons: {val_singletons:,} ({val_singletons/len(val_gt_dict)*100:.1f}%)")

# Pre-extract arrays for ultra-fast vectorized evaluation
arr_val_s1 = val_df['s1_id'].values
arr_val_s2 = val_df['s2s3_id'].values
arr_val_prob = val_df['prob'].values


def evaluate_f05_fast(threshold):
    """Calculates true macro F0.5 across all validation entities in < 0.5s."""
    mask = arr_val_prob >= threshold
    active_s1 = arr_val_s1[mask]
    active_s2 = arr_val_s2[mask]
    
    pred_dict = defaultdict(set)
    for i in range(len(active_s1)):
        pred_dict[active_s1[i]].add(active_s2[i])
        
    scores = []
    for sid, true_set in val_gt_dict.items():
        p_set = pred_dict.get(sid, set())
        # Enforce max 11 matches per entity
        if len(p_set) > 11:
            p_set = set(list(p_set)[:11])
            
        if len(true_set) == 0 and len(p_set) == 0:
            scores.append(1.0)
        elif len(true_set) == 0 or len(p_set) == 0:
            scores.append(0.0)
        else:
            tp = len(true_set & p_set)
            prec = tp / len(p_set)
            rec = tp / len(true_set)
            if prec + rec > 0:
                scores.append((1.25 * prec * rec) / (0.25 * prec + rec))
            else:
                scores.append(0.0)
    return np.mean(scores), int(mask.sum())


# Coarse grid sweep
coarse_grid = np.arange(0.10, 0.95, 0.05)
best_threshold = 0.50
best_f05 = 0.0

print("\n--- Phase 1: Coarse Threshold Sweep (0.10 to 0.90) ---")
for thr in coarse_grid:
    score, pairs = evaluate_f05_fast(thr)
    is_best = score > best_f05
    if is_best:
        best_f05 = score
        best_threshold = thr
    marker = " ◀ BEST" if is_best else ""
    print(f"  Threshold {thr:.2f} | Positive Pairs: {pairs:8,} | Macro F0.5: {score:.5f}{marker}")

# Fine grid sweep around best coarse point
fine_grid = np.arange(max(0.02, best_threshold - 0.08), min(0.98, best_threshold + 0.08), 0.01)
print(f"\n--- Phase 2: Fine Threshold Sweep (Around {best_threshold:.2f}, step 0.01) ---")
for thr in fine_grid:
    score, pairs = evaluate_f05_fast(thr)
    if score > best_f05:
        best_f05 = score
        best_threshold = thr
        print(f"  Threshold {thr:.2f} | Positive Pairs: {pairs:8,} | Macro F0.5: {score:.5f} ◀ NEW BEST")

print("\n" + "=" * 80)
print(f"🏆 OPTIMAL DECISION THRESHOLD: {best_threshold:.2f}")
print(f"🏆 VALIDATION MACRO F0.5 SCORE: {best_f05:.5f}")
print("=" * 80)


In [ ]:


# ==============================================================================
# === CELL 11: Memory Reclamation & Test Multilingual Embeddings
# ==============================================================================

print("\nReclaiming training memory before full test inference...")
# Safely delete large training objects to free RAM/VRAM for test phase
for _var in ['train_candidates', 'train_features', 'sub_s1_embeddings', 'train_s2s3_embeddings',
             'X', 'y', 'X_train', 'X_val', 'y_train', 'y_val', 'dtrain', 'dval', 'val_df',
             'train_s2s3', 'labels', 's1_vec', 's2_vec', 'sub_s1_emb_map', 'train_s2s3_emb_map',
             'arr_val_s1', 'arr_val_s2', 'arr_val_prob', 'val_probs', 'val_gt_dict',
             'train_sub_gt', 'train_sub_s1']:
    if _var in globals():
        try:
            del globals()[_var]
        except KeyError:
            pass
gc.collect()
if CUDA_AVAILABLE:
    torch.cuda.empty_cache()
print("✅ Training memory freed.")

# Reclaim 15.85 GB disk space: train_s2s3_emb.npy is no longer needed after training
train_s2s3_emb_file = os.path.join(OUTPUT_DIR, "train_s2s3_emb.npy")
if os.path.exists(train_s2s3_emb_file):
    print(f"🗑️ Reclaiming disk: removing {train_s2s3_emb_file} (frees 15.85 GB disk space)...")
    try:
        os.remove(train_s2s3_emb_file)
        if os.path.exists(train_s2s3_emb_file + ".progress"):
            os.remove(train_s2s3_emb_file + ".progress")
        print("✅ Disk space reclaimed successfully.")
    except Exception as e:
        print(f"  Note: could not delete {train_s2s3_emb_file}: {e}")


print("\nLoading and preprocessing Test Sets on-demand...")
t_test_load = time.time()
test_s1 = preprocess_df(pd.read_csv(find_file("test_source1.tsv"), sep="\t", dtype=str), "Test S1")
test_s2 = preprocess_df(pd.read_csv(find_file("test_source2.tsv"), sep="\t", dtype=str), "Test S2")
test_s3 = preprocess_df(pd.read_csv(find_file("test_source3.tsv"), sep="\t", dtype=str), "Test S3")
print(f"✅ Test datasets loaded in {fmt_time(time.time()-t_test_load)}")
print(f"   Test S1:  {len(test_s1):,}  | Test S2:  {len(test_s2):,}  | Test S3:  {len(test_s3):,}")

test_s2s3 = pd.concat([test_s2, test_s3], ignore_index=True)
# Free test_s2 and test_s3 immediately
del test_s2, test_s3
gc.collect()

test_s1_emb_path = os.path.join(OUTPUT_DIR, "test_s1_emb.npy")
test_s2s3_emb_path = os.path.join(OUTPUT_DIR, "test_s2s3_emb.npy")
test_batch = 256 if IS_KAGGLE else 512

if os.path.exists(test_s1_emb_path) and not os.path.exists(test_s1_emb_path + ".progress"):
    print(f"Loading cached {test_s1_emb_path}...")
    test_s1_embeddings = np.load(test_s1_emb_path)
else:
    test_s1_embeddings = encode_texts_to_disk(test_s1['embed_text'], test_s1_emb_path, desc="Test S1", batch_size=test_batch)

test_s2s3_progress = test_s2s3_emb_path + ".progress"
if os.path.exists(test_s2s3_emb_path) and not os.path.exists(test_s2s3_progress):
    print(f"Loading cached {test_s2s3_emb_path} (mmap_mode='r')...")
    test_s2s3_embeddings = np.load(test_s2s3_emb_path, mmap_mode='r')
else:
    test_s2s3_embeddings = encode_texts_to_disk(test_s2s3['embed_text'], test_s2s3_emb_path, desc="Test S2+S3", batch_size=test_batch)



In [ ]:

# ==============================================================================
# === CELL 12: Test Blocking (100% Test Set, Strict Top-K Candidate Pruning)
# ==============================================================================

print("\n" + "=" * 80)
print("BLOCKING ON TEST DATA (FULL 100% UNIVERSE)")
print("=" * 80)

test_cand_path = os.path.join(OUTPUT_DIR, "test_candidates_v4.csv")
if os.path.exists(test_cand_path):
    print(f"Loading cached {test_cand_path}...")
    test_candidates = pd.read_csv(test_cand_path)
else:
    test_candidates = generate_candidate_universe(
        test_s1, test_s1_embeddings, test_s2s3, test_s2s3_embeddings,
        max_candidates_per_s1=10
    )
    test_candidates.to_csv(test_cand_path, index=False)
    print(f"💾 Checkpoint saved: {test_cand_path}")

avg_test_cands = len(test_candidates) / len(test_s1)
print(f"Test Candidate Universe: {len(test_candidates):,} pairs across {len(test_s1):,} S1 entities (Avg {avg_test_cands:.2f} pairs/S1)")


In [ ]:


# ==============================================================================
# === CELL 13: Test Feature Extraction, Inference & Submission Generation
# ==============================================================================

test_s1_map = precompute_embedding_index(test_s1)
test_s2s3_map = precompute_embedding_index(test_s2s3)

test_features_path = os.path.join(OUTPUT_DIR, "test_features_v4.csv")
if os.path.exists(test_features_path):
    print(f"\nLoading cached {test_features_path}...")
    test_features = pd.read_csv(test_features_path)
else:
    test_features = extract_features_pipeline(
        test_candidates, test_s1, test_s2s3,
        test_s1_embeddings, test_s2s3_embeddings,
        test_s1_map, test_s2s3_map, desc="test"
    )
    # Disk space check before saving (~20 cols × 4 bytes × N rows + CSV overhead)
    est_size_mb = len(test_features) * 20 * 6 / 1e6  # conservative CSV estimate
    print(f"  Estimated CSV size: {est_size_mb:.0f} MB")
    if est_size_mb < 15000:  # Kaggle /kaggle/working has ~20GB
        test_features.to_csv(test_features_path, index=False, chunksize=100000)
        print(f"💾 Checkpoint saved: {test_features_path}")
    else:
        print(f"⚠️ Skipping CSV save (too large: {est_size_mb:.0f} MB). Continuing in-memory.")

print(f"\nRunning LightGBM inference on {len(test_features):,} test candidate pairs...")
t_inf = time.time()
# Only keep feature columns that exist in test_features (guards against cached CSV column drift)
avail_feature_cols = [c for c in feature_cols if c in test_features.columns]
if len(avail_feature_cols) < len(feature_cols):
    missing = set(feature_cols) - set(avail_feature_cols)
    print(f"  ⚠️ Missing {len(missing)} feature columns in test set: {missing}")
    print(f"  Imputing missing columns with 0.0...")
    for mc in missing:
        test_features[mc] = 0.0
X_test = test_features[feature_cols].values.astype(np.float32)
test_probs = model.predict(X_test)
test_features['prob'] = test_probs
print(f"✅ Inference finished in {fmt_time(time.time()-t_inf)}")

# Apply optimal threshold and sort by highest confidence
final_matches = test_features[test_features['prob'] >= best_threshold].sort_values('prob', ascending=False)
print(f"\nPrediction Breakdown:")
print(f"  Total Candidate Pairs: {len(test_features):,}")
print(f"  Pairs >= Optimal Threshold ({best_threshold:.2f}): {len(final_matches):,} ({len(final_matches)/len(test_features)*100:.2f}%)")

# Fast dictionary aggregation for output generation (Instant, no iterrows)
print("\nGenerating final submission files...")
t_sub = time.time()

# 1. Final matching predictions
match_dict = defaultdict(list)
for sid, mid in zip(final_matches['s1_id'].values, final_matches['s2s3_id'].values):
    match_dict[sid].append(mid)

all_test_s1_ids = test_s1['entity_id'].values

matching_rows = []
for sid in all_test_s1_ids:
    matched = list(dict.fromkeys(match_dict.get(sid, [])))
    # Cap at 11 matches per entity
    if len(matched) > 11:
        matched = matched[:11]
    matching_rows.append({
        'source1_entity_id': sid,
        'matched_entity_ids': ','.join(matched) if matched else ''
    })

match_out_file = os.path.join(OUTPUT_DIR, "matching_results.tsv")
df_match_out = pd.DataFrame(matching_rows)
# PDF rule: matched_entity_ids must never be NaN — always empty string for singletons
df_match_out['matched_entity_ids'] = df_match_out['matched_entity_ids'].fillna('')
df_match_out.to_csv(match_out_file, sep='\t', index=False)

# 2. Candidate generation submission (Critically evaluated by organizers!)
cand_dict = defaultdict(list)
for sid, mid in zip(test_candidates['s1_id'].values, test_candidates['s2s3_id'].values):
    cand_dict[sid].append(mid)

candidate_rows = []
for sid in all_test_s1_ids:
    cands = list(dict.fromkeys(cand_dict.get(sid, [])))
    if len(cands) > 10:
        cands = cands[:10]
    candidate_rows.append({
        'source1_entity_id': sid,
        'candidate_entity_ids': ','.join(cands) if cands else ''
    })

cand_out_file = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
df_cand_out = pd.DataFrame(candidate_rows)
df_cand_out['candidate_entity_ids'] = df_cand_out['candidate_entity_ids'].fillna('')
df_cand_out.to_csv(cand_out_file, sep='\t', index=False)

print(f"✅ Both submission files written in {time.time()-t_sub:.1f}s")



In [ ]:

# ==============================================================================
# === CELL 14: Final Submission Verification & Audit
# ==============================================================================

print("\n" + "=" * 80)
print("AUDITING FINAL SUBMISSION FILES")
print("=" * 80)

df_match_check = pd.read_csv(match_out_file, sep='\t', dtype=str).fillna('')
df_cand_check = pd.read_csv(cand_out_file, sep='\t', dtype=str).fillna('')

n_test_total = len(all_test_s1_ids)
assert len(df_match_check) == n_test_total, f"Row count mismatch in matching_results: {len(df_match_check)} vs {n_test_total}"
assert len(df_cand_check) == n_test_total, f"Row count mismatch in candidate_pairs: {len(df_cand_check)} vs {n_test_total}"

# PDF rule: No duplicate source1_entity_id rows
dup_match = df_match_check['source1_entity_id'].duplicated().sum()
dup_cand = df_cand_check['source1_entity_id'].duplicated().sum()
assert dup_match == 0, f"DUPLICATE source1_entity_id rows in matching_results.tsv: {dup_match}"
assert dup_cand == 0, f"DUPLICATE source1_entity_id rows in candidate_pairs.tsv: {dup_cand}"

# PDF rule: No S1 IDs in matched_entity_ids (self-match check)
all_s1_id_set = set(test_s1['entity_id'].values)
all_valid_s2s3_ids = set(test_s2s3['entity_id'].values)
self_match_count = 0
bad_id_count = 0
for m in df_match_check['matched_entity_ids']:
    if m.strip() == '':
        continue
    for mid in m.split(','):
        mid = mid.strip()
        if mid in all_s1_id_set:
            self_match_count += 1
        if mid not in all_valid_s2s3_ids:
            bad_id_count += 1

if self_match_count > 0:
    print(f"  ❌ CRITICAL: {self_match_count} self-matches (S1 IDs) found in matched_entity_ids! These will be REJECTED.")
else:
    print(f"  ✅ No self-matches found in matched_entity_ids.")

if bad_id_count > 0:
    print(f"  ❌ CRITICAL: {bad_id_count} IDs in matched_entity_ids do not exist in test S2/S3! These will be REJECTED.")
else:
    print(f"  ✅ All matched IDs verified to exist in test S2/S3.")

# PDF rule: Every matched ID must also appear in candidate_pairs.tsv
cand_set_check = set()
for sid, clist in zip(df_cand_check['source1_entity_id'], df_cand_check['candidate_entity_ids']):
    if clist.strip():
        for c in clist.split(','):
            cand_set_check.add((sid.strip(), c.strip()))

match_not_in_cands = 0
for sid, mlist in zip(df_match_check['source1_entity_id'], df_match_check['matched_entity_ids']):
    if mlist.strip():
        for m in mlist.split(','):
            if (sid.strip(), m.strip()) not in cand_set_check:
                match_not_in_cands += 1

if match_not_in_cands > 0:
    print(f"  ⚠️  WARNING: {match_not_in_cands} matched IDs do not appear in candidate_pairs.tsv (pipeline bug signal).")
else:
    print(f"  ✅ All matched IDs are a subset of candidate_pairs — pipeline is consistent.")

# Summary statistics
n_with_match = sum(1 for m in df_match_check['matched_entity_ids'] if m.strip() != '')
n_singletons = n_test_total - n_with_match
total_pred_matches = sum(len(m.split(',')) for m in df_match_check['matched_entity_ids'] if m.strip() != '')
total_cands = sum(len(c.split(',')) for c in df_cand_check['candidate_entity_ids'] if c.strip() != '')
avg_cands_final = total_cands / max(n_test_total, 1)

print(f"\n📊 FINAL SUBMISSION AUDIT REPORT:")
print(f"   Total S1 Records: {n_test_total:,}")
print(f"   Entities with Matches: {n_with_match:,} ({n_with_match/n_test_total*100:.2f}%)")
print(f"   Predicted Singletons (Empty matches): {n_singletons:,} ({n_singletons/n_test_total*100:.2f}%)")
print(f"   Total Matches Predicted: {total_pred_matches:,} (Avg {total_pred_matches/max(n_with_match, 1):.2f}/entity)")
print(f"   Candidate Pairs Generated: {total_cands:,} (Avg {avg_cands_final:.2f} candidates/entity)")
print(f"   Validation Macro F0.5 Score: {best_f05:.5f}")
print("=" * 80)
print("🎯 SUBMISSION READY FILES IN:")
print(f"   1. {match_out_file}  <-- Upload to Leaderboard (matching_results.tsv)")
print(f"   2. {cand_out_file}   <-- Candidate Generation Evaluation (candidate_pairs.tsv)")
print("=" * 80)

# ------------------------------------------------------------------------------
# Final Disk Optimization: Clean up large temporary embedding matrices
# ------------------------------------------------------------------------------
# Deleting test_s2s3_emb.npy (15.85 GB) keeps output directory under ~300 MB,
# guaranteeing instant notebook commit and compliance with Kaggle's 20 GB output limit.
print("\nCleaning up large temporary embedding files to free disk space...")
for _emb_file in ["test_s2s3_emb.npy", "test_s1_emb.npy", "train_sub_s1_emb.npy"]:
    _p = os.path.join(OUTPUT_DIR, _emb_file)
    if os.path.exists(_p):
        try:
            os.remove(_p)
            if os.path.exists(_p + ".progress"):
                os.remove(_p + ".progress")
            print(f"  🗑️ Removed temporary file: {_emb_file}")
        except Exception as e:
            pass

print("\n" + "=" * 80)
print("📦 FINAL OUTPUT DIRECTORY INVENTORY (/kaggle/working):")
print("=" * 80)
total_disk_mb = 0
for _f in sorted(os.listdir(OUTPUT_DIR)):
    _fp = os.path.join(OUTPUT_DIR, _f)
    if os.path.isfile(_fp):
        _sz_mb = os.path.getsize(_fp) / (1024 * 1024)
        total_disk_mb += _sz_mb
        print(f"  📁 {_f:32s}: {_sz_mb:8.2f} MB")
print("-" * 80)
print(f"  TOTAL OUTPUT SIZE: {total_disk_mb:.2f} MB ({total_disk_mb/1024:.2f} GB) / 20.0 GB Kaggle Limit ✅")
print("=" * 80)

